## Задание
Допишите реализацию класса `MyMultiHeadAttention` по указанному шаблону. Убедитесь, что при `d_model=64`, `num_heads=8` и входе `(10, 2, 64)` выход имеет ту же форму `(10, 2, 64)`.

Выполните задание локально, а затем сверьтесь с авторским решением. 

In [1]:
import torch
import torch.nn as nn

In [4]:
class MyMultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model должно делиться на num_heads"
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        # Линейные слои для Q, K, V и выходной слой
        self.WQ = nn.Linear(d_model, d_model, bias=False)
        self.WK = nn.Linear(d_model, d_model, bias=False)
        self.WV = nn.Linear(d_model, d_model, bias=False)
        self.WO = nn.Linear(d_model, d_model, bias=False)

    def forward(self, query, key, value, mask=None):
        # query, key, value: (seq_len, batch, d_model)
        seq_len, batch, d_model = query.shape

        # 1) Линейные проекции в Q, K, V
        Q = self.WQ(query)   # (seq_len, batch, d_model)
        K = self.WK(key)     # (seq_len, batch, d_model)
        V = self.WV(value)   # (seq_len, batch, d_model)

        # 2) Переупорядочивание размерностей
        #    (seq_len, batch, d_model) → (batch, seq_len, d_model)
        Q = Q.permute(1, 0, 2)
        K = K.permute(1, 0, 2)
        V = V.permute(1, 0, 2)

        # 3) Разбиение на H голов:
        #    (batch, seq_len, d_model)
        #  → (batch, seq_len, H, d_head)
        #  → (batch, H, seq_len, d_head)
        Q = Q.view(batch, seq_len, self.num_heads, self.head_dim) \
             .permute(0, 2, 1, 3)
        K = K.view(batch, seq_len, self.num_heads, self.head_dim) \
             .permute(0, 2, 1, 3)
        V = V.view(batch, seq_len, self.num_heads, self.head_dim) \
             .permute(0, 2, 1, 3)
        
        # 4) Скалярное произведение и масштабирование
        #    (batch, H, seq, d_head) × (batch, H, d_head, seq)
        #  → (batch, H, seq, seq)
        scores = torch.matmul(Q, K.transpose(-2, -1)) \
               / (self.head_dim ** 0.5) 
        
        # 5) Применение маски (если передана)
        if mask is not None:
            # (seq_len, seq_len) → (1, 1, seq_len, seq_len)
            mask = mask.unsqueeze(0).unsqueeze(1)
            scores = scores.masked_fill(mask == 0, float('-inf')) 

        # 6) Softmax → веса внимания
        attention_weights = torch.softmax(scores, dim=-1)
        
        # 7) Взвешенное суммирование по V
        # (batch, H, seq, seq) × (batch, H, seq, d_head)
        heads_output = torch.matmul(attention_weights, V) 

        # 8) Конкатенация голов:
        #    (batch, H, seq, d_head)
        #  → (batch, seq, H, d_head)
        #  → (batch, seq, d_model)
        concat = heads_output.permute(0, 2, 1, 3) \
                             .contiguous() \
                             .view(batch, seq_len, d_model)

        # 9) Финальная линейная проекция
        output = self.WO(concat)  # (batch, seq_len, d_model) 

        return output.permute(1, 0, 2) 

# Использование класса MyMultiHeadAttention:
d_model, num_heads = 64, 8
mha = MyMultiHeadAttention(d_model, num_heads)
x = torch.randn(10, 2, d_model)    # (seq_len=10, batch=2, d_model=64)
y = mha(x, x, x, mask=None)
print(y.shape)  # (10, 2, 64)

torch.Size([10, 2, 64])
